# FreshCart AI: Data Preprocessing
Building item sequences, vocabulary, and train/val/test splits for the LSTM model.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from collections import Counter

DATA_DIR = "../data/processed"
MAX_LEN = 100      # Maximum sequence length -- truncate + pad to this length
VOCAB_SIZE = 5000  # Number of real items (index 0 reserved for PAD)
PAD_TOKEN = 0      # Padding token index

print(f"Config: MAX_LEN={MAX_LEN}, VOCAB_SIZE={VOCAB_SIZE}, PAD_TOKEN={PAD_TOKEN}")

In [ ]:
orders = pd.read_csv(f"{DATA_DIR}/orders_subset.csv")
order_products = pd.read_csv(f"{DATA_DIR}/order_products_subset.csv")

# Keep only 'prior' eval_set orders for sequence building
orders_prior = orders[orders['eval_set'] == 'prior'].copy()

print(f"Prior orders: {orders_prior.shape[0]:,}")
print(f"Prior order-product pairs: {order_products.shape[0]:,}")
print(f"Unique users: {orders_prior['user_id'].nunique():,}")

## Step 1: Build Per-User Item Sequences

In [ ]:
# Merge order_products with orders to get user_id and order_number per product
order_details = order_products.merge(
    orders_prior[['order_id', 'user_id', 'order_number']],
    on='order_id'
)

# Sort by user_id then order_number to ensure chronological order
order_details = order_details.sort_values(['user_id', 'order_number', 'add_to_cart_order'])

# Build per-user sequences: list of product_ids in purchase order
user_sequences_raw = (
    order_details
    .groupby('user_id')['product_id']
    .apply(list)
    .to_dict()
)

# Summary
seq_lengths = [len(v) for v in user_sequences_raw.values()]
print(f"Total users with sequences: {len(user_sequences_raw):,}")
print(f"Min sequence length: {min(seq_lengths)}")
print(f"Max sequence length: {max(seq_lengths)}")
print(f"Mean sequence length: {np.mean(seq_lengths):.1f}")
print(f"Sequences with length > {MAX_LEN}: {sum(l > MAX_LEN for l in seq_lengths)} ({sum(l > MAX_LEN for l in seq_lengths)/len(seq_lengths)*100:.1f}%)")

**Strategy:** By sorting on `order_number` then `add_to_cart_order`, we preserve the temporal structure that the LSTM will learn from. Each user's sequence represents their full purchase history in chronological order — the model learns to predict the next item from the sequence of previous purchases.

## Step 2: Build Vocabulary (Top 5,000 by Total Frequency)

In [ ]:
# Count total occurrences of each product_id across ALL prior orders
all_products = order_products['product_id'].tolist()
product_counts = Counter(all_products)

print(f"Total unique products in dataset: {len(product_counts):,}")

# Take top VOCAB_SIZE products by frequency
top_products = product_counts.most_common(VOCAB_SIZE)

# Build vocab dict: product_id -> index (1-indexed; 0 is reserved for PAD)
# Key "PAD" maps to 0
vocab = {"PAD": PAD_TOKEN}
for idx, (product_id, count) in enumerate(top_products, start=1):
    vocab[str(product_id)] = idx

print(f"Vocabulary size (including PAD): {len(vocab):,}")
print(f"Expected: {VOCAB_SIZE + 1} (5,000 products + 1 PAD token)")
assert len(vocab) == VOCAB_SIZE + 1, f"Vocab size mismatch: {len(vocab)} != {VOCAB_SIZE + 1}"

# Save vocab.json
vocab_path = f"{DATA_DIR}/vocab.json"
with open(vocab_path, 'w') as f:
    json.dump(vocab, f)
print(f"Saved: {vocab_path}")

# Spot-check
print(f"Most frequent product: product_id={top_products[0][0]}, count={top_products[0][1]:,}, index=1")
print(f"Least frequent in vocab: product_id={top_products[-1][0]}, count={top_products[-1][1]:,}, index={VOCAB_SIZE}")

## Step 3: Map Sequences to Integer Indices + Pad/Truncate

In [ ]:
def encode_and_pad(product_id_list, vocab, max_len, pad_token=0):
    """
    Map product_ids to vocab indices, truncate to max_len (keep LAST max_len items),
    then pre-pad on the left with pad_token to reach max_len.
    Products not in vocab are skipped (treated as unknown).
    """
    encoded = [vocab[str(pid)] for pid in product_id_list if str(pid) in vocab]
    # Truncate from the START (keep the most recent MAX_LEN items)
    if len(encoded) > max_len:
        encoded = encoded[-max_len:]
    # Pre-pad on the left
    padded = [pad_token] * (max_len - len(encoded)) + encoded
    return padded

# Map all user sequences
user_sequences_encoded = {}
for user_id, product_list in user_sequences_raw.items():
    user_sequences_encoded[user_id] = encode_and_pad(product_list, vocab, MAX_LEN, PAD_TOKEN)

# Convert to numpy array: shape (n_users, MAX_LEN)
user_ids_list = list(user_sequences_encoded.keys())
sequences_array = np.array([user_sequences_encoded[uid] for uid in user_ids_list], dtype=np.int32)

print(f"Sequences array shape: {sequences_array.shape}")
print(f"Expected: ({len(user_ids_list)}, {MAX_LEN})")
assert sequences_array.shape[1] == MAX_LEN, f"Sequence length mismatch: {sequences_array.shape[1]} != {MAX_LEN}"
print(f"Dtype: {sequences_array.dtype}")
print(f"Min value (should be 0=PAD): {sequences_array.min()}")
print(f"Max value (should be <= {VOCAB_SIZE}): {sequences_array.max()}")

# Save intermediate sequences for use in Part 2 (splitting)
np.savez_compressed(f"{DATA_DIR}/sequences.npz",
                    sequences=sequences_array,
                    user_ids=np.array(user_ids_list))
print(f"Saved: {DATA_DIR}/sequences.npz")

**Pre-padding strategy:** Padding on the left means the LSTM sees real data at the end of each sequence — this is critical. Since the LSTM processes sequences left-to-right, placing real items at the end ensures the hidden state captures the most recent purchasing context before making predictions. The most recent purchases should have the highest influence on the next recommendation.

---

## Part 2: Train / Validation / Test Splits
Split by user (80% train, 10% val, 10% test). No user appears in more than one split.

In [ ]:
from sklearn.model_selection import train_test_split

# Load sequences and user_ids saved by Part 1
data = np.load(f"{DATA_DIR}/sequences.npz")
sequences_array = data['sequences']
user_ids_array = data['user_ids']

print(f"Loaded sequences shape: {sequences_array.shape}")
print(f"Total users: {len(user_ids_array):,}")
print(f"Sequence length: {sequences_array.shape[1]}")

RANDOM_STATE = 42
VAL_TEST_SIZE = 0.20   # 20% held out for val+test
TEST_FRACTION = 0.50   # 50% of val+test becomes test -> 10% overall

In [ ]:
# Step 1: Split user indices into train (80%) and temp (20%)
all_indices = np.arange(len(user_ids_array))

train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=VAL_TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

# Step 2: Split temp (20%) into val (10%) and test (10%)
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    shuffle=True
)

# Extract sequences for each split
X_train = sequences_array[train_indices]
X_val   = sequences_array[val_indices]
X_test  = sequences_array[test_indices]

train_user_ids = user_ids_array[train_indices]
val_user_ids   = user_ids_array[val_indices]
test_user_ids  = user_ids_array[test_indices]

total = len(user_ids_array)
print(f"Train:      {len(X_train):,} users ({len(X_train)/total*100:.1f}%)")
print(f"Validation: {len(X_val):,}   users ({len(X_val)/total*100:.1f}%)")
print(f"Test:       {len(X_test):,}  users ({len(X_test)/total*100:.1f}%)")
print(f"Total:      {len(X_train)+len(X_val)+len(X_test):,} (should equal {total:,})")

In [ ]:
# Verify no user appears in more than one split
train_set = set(train_user_ids)
val_set   = set(val_user_ids)
test_set  = set(test_user_ids)

assert len(train_set & val_set) == 0,   "DATA LEAK: user overlap between train and val!"
assert len(train_set & test_set) == 0,  "DATA LEAK: user overlap between train and test!"
assert len(val_set & test_set) == 0,    "DATA LEAK: user overlap between val and test!"
assert len(train_set) + len(val_set) + len(test_set) == total, "Split sizes don't add up!"

print("No user overlap between splits -- data leakage check passed")
print(f"Split sizes sum to {total:,} -- all users accounted for")

In [ ]:
# Save splits
np.savez_compressed(f"{DATA_DIR}/X_train.npz", sequences=X_train, user_ids=train_user_ids)
np.savez_compressed(f"{DATA_DIR}/X_val.npz",   sequences=X_val,   user_ids=val_user_ids)
np.savez_compressed(f"{DATA_DIR}/X_test.npz",  sequences=X_test,  user_ids=test_user_ids)

print(f"Saved: {DATA_DIR}/X_train.npz -- shape {X_train.shape}")
print(f"Saved: {DATA_DIR}/X_val.npz   -- shape {X_val.shape}")
print(f"Saved: {DATA_DIR}/X_test.npz  -- shape {X_test.shape}")

In [ ]:
# Verify: load and confirm arrays are intact
check_train = np.load(f"{DATA_DIR}/X_train.npz")
check_val   = np.load(f"{DATA_DIR}/X_val.npz")
check_test  = np.load(f"{DATA_DIR}/X_test.npz")

assert check_train['sequences'].shape == X_train.shape, "X_train shape mismatch after reload"
assert check_val['sequences'].shape   == X_val.shape,   "X_val shape mismatch after reload"
assert check_test['sequences'].shape  == X_test.shape,  "X_test shape mismatch after reload"
assert check_train['sequences'].dtype == np.int32,      "X_train dtype should be int32"

print("All .npz files load correctly with expected shapes and dtype")
print(f"  X_train: {check_train['sequences'].shape}")
print(f"  X_val:   {check_val['sequences'].shape}")
print(f"  X_test:  {check_test['sequences'].shape}")

## Summary

Preprocessing complete. The following files are ready for model training:

| File | Content | Shape |
|------|---------|-------|
| `vocab.json` | Product ID to integer index mapping | 5,001 entries |
| `X_train.npz` | Training sequences (padded integer arrays) | (~8000, 100) |
| `X_val.npz` | Validation sequences | (~1000, 100) |
| `X_test.npz` | Test sequences | (~1000, 100) |

All users appear in exactly one split. Same user's orders do not appear across splits.